# TP 5 — Projet de synthèse : carnet de notes

In [1]:
from datetime import datetime
import json
from functools import wraps

etudiants = {
    "L3-001": {
        "nom": "Rakoto", "prenom": "Soa",
        "notes": {"Python": [15, 14], "Réseaux": [12]},
    },
    "L3-002": {
        "nom": "Rabe", "prenom": "Andry",
        "notes": {"Python": [9, 13], "Réseaux": [11, 10]},
    },
}
historique = []

# Décorateur de journalisation
def journaliser(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        historique.append((
            datetime.now().isoformat(timespec="seconds"),
            fonction.__name__,
            {"args": args, "kwargs": kwargs}
        ))
        return fonction(*args, **kwargs)
    return enveloppe

# Vérification du matricule en premier argument
def etudiant_existe(fonction):
    @wraps(fonction)
    def enveloppe(matricule, *args, **kwargs):
        if matricule not in etudiants:
            raise KeyError(f"Matricule inconnu : {matricule}")
        return fonction(matricule, *args, **kwargs)
    return enveloppe

# Validation de la note en troisième argument
def note_valide(fonction):
    @wraps(fonction)
    def enveloppe(matricule, matiere, note, *args, **kwargs):
        if not isinstance(note, (int, float)) or not 0 <= note <= 20:
            raise ValueError("La note doit être comprise entre 0 et 20.")
        return fonction(matricule, matiere, note, *args, **kwargs)
    return enveloppe

@journaliser
def ajouter_etudiant(matricule, nom, prenom):
    if matricule in etudiants:
        return False, f"Matricule déjà utilisé : {matricule}"
    etudiants[matricule] = {"nom": nom, "prenom": prenom, "notes": {}}
    return True, f"Étudiant {prenom} {nom} ajouté."

@journaliser
@etudiant_existe
@note_valide
def ajouter_note(matricule, matiere, note):
    etudiants[matricule]["notes"].setdefault(matiere, []).append(note)
    return True, f"Note {note}/20 ajoutée en {matiere}."

@etudiant_existe
def moyenne_matiere(matricule, matiere):
    notes = etudiants[matricule]["notes"].get(matiere, [])
    return sum(notes) / len(notes) if notes else None

@etudiant_existe
def moyenne_generale(matricule):
    moyennes = [sum(ns) / len(ns) for ns in etudiants[matricule]["notes"].values() if ns]
    return sum(moyennes) / len(moyennes) if moyennes else 0.0

def classement():
    liste = []
    for matricule, e in etudiants.items():
        nom_complet = f"{e['prenom']} {e['nom']}"
        liste.append((nom_complet, moyenne_generale(matricule)))
    return sorted(liste, key=lambda x: x[1], reverse=True)

def majors():
    meilleurs = {}
    for matricule, e in etudiants.items():
        nom = f"{e['prenom']} {e['nom']}"
        for matiere in e["notes"]:
            moyenne = moyenne_matiere(matricule, matiere)
            if moyenne is not None and (matiere not in meilleurs or moyenne > meilleurs[matiere][1]):
                meilleurs[matiere] = (nom, moyenne)
    return {matiere: infos[0] for matiere, infos in meilleurs.items()}

def sauvegarder(fichier):
    with open(fichier, "w", encoding="utf-8") as flux:
        json.dump(etudiants, flux, ensure_ascii=False, indent=2)
    return f"Sauvegardé dans {fichier}"

def charger(fichier):
    global etudiants
    with open(fichier, "r", encoding="utf-8") as flux:
        etudiants = json.load(flux)
    return f"Chargé depuis {fichier}"

def afficher_bulletin(matricule):
    if matricule not in etudiants:
        print("Matricule inconnu.")
        return
    e = etudiants[matricule]
    print(f"\nBulletin de {e['prenom']} {e['nom']} — {matricule}")
    for matiere, ns in e["notes"].items():
        moy = sum(ns) / len(ns) if ns else 0
        print(f"{matiere} : {ns} ; moyenne = {moy:.2f}")
    print(f"Moyenne générale : {moyenne_generale(matricule):.2f}/20")

print("Classement initial :")
for rang_etudiant, (nom, moy) in enumerate(classement(), start=1):
    print(f"{rang_etudiant}. {nom:<20} {moy:.2f}")
print("Majors :", majors())
afficher_bulletin("L3-001")

Classement initial :
1. Soa Rakoto           13.25
2. Andry Rabe           10.75
Majors : {'Python': 'Soa Rakoto', 'Réseaux': 'Soa Rakoto'}

Bulletin de Soa Rakoto — L3-001
Python : [15, 14] ; moyenne = 14.50
Réseaux : [12] ; moyenne = 12.00
Moyenne générale : 13.25/20


# Menu interactif du TP 5

In [ ]:
def menu():
    while True:
        print("\n===== CARNET DE NOTES =====")
        print("1. Ajouter un étudiant")
        print("2. Ajouter une note")
        print("3. Afficher un bulletin")
        print("4. Afficher le classement")
        print("5. Afficher les majors")
        print("6. Afficher l'historique")
        print("7. Sauvegarder")
        print("8. Charger")
        print("0. Quitter")
        choix = input("Votre choix : ").strip()
        try:
            if choix == "1":
                print(ajouter_etudiant(input("Matricule : ").strip(),
                                       input("Nom : ").strip(),
                                       input("Prénom : ").strip()))
            elif choix == "2":
                matricule = input("Matricule : ").strip()
                matiere = input("Matière : ").strip()
                note = float(input("Note /20 : "))
                print(ajouter_note(matricule, matiere, note))
            elif choix == "3":
                afficher_bulletin(input("Matricule : ").strip())
            elif choix == "4":
                for i, (nom, moy) in enumerate(classement(), start=1):
                    print(f"{i}. {nom:<20} {moy:.2f}")
            elif choix == "5":
                for matiere, nom in majors().items():
                    print(f"{matiere} : {nom}")
            elif choix == "6":
                for ligne in historique:
                    print(ligne)
            elif choix == "7":
                fichier = input("Fichier [carnet_notes.json] : ").strip() or "carnet_notes.json"
                print(sauvegarder(fichier))
            elif choix == "8":
                fichier = input("Fichier [carnet_notes.json] : ").strip() or "carnet_notes.json"
                print(charger(fichier))
            elif choix == "0":
                print("Au revoir !")
                break
            else:
                print("Choix invalide.")
        except (ValueError, KeyError, OSError) as erreur:
            print("Erreur :", erreur)
            
# Pour démarrer le menu, exécuter dans une cellule : menu()
menu()



===== CARNET DE NOTES =====
1. Ajouter un étudiant
2. Ajouter une note
3. Afficher un bulletin
4. Afficher le classement
5. Afficher les majors
6. Afficher l'historique
7. Sauvegarder
8. Charger
0. Quitter
